# BPIC15 RL Pipeline — Concrete Notebook Plan

This notebook is the **implementation roadmap** from process-graph analysis to a first PPO baseline.
It assumes work in `process_graph.ipynb` is complete through:
- data loading (`logs`, `MUNICIPALITIES`)
- DFGs (`dfgs`)
- temporal activity ranks (`act_ranks`)
- cross-municipality transition comparisons

---

## 0) Scope, assumptions, and outputs
**Goal:** Build a reproducible baseline pipeline for case-level decision optimization.

**Primary inputs**
- `./dataset/BPIC15_Municipality{1..5}.jsonocel`
- optional carry-over artifacts from `process_graph.ipynb` (edge frequencies, `act_ranks`)

**Primary outputs (write to `./output`)**
1. `case_step_features.parquet`
2. `transition_stats.csv`
3. `duration_stats.csv`
4. `valid_action_space.csv` (or refreshed version)
5. `sim_calibration.json`
6. `sim_validation_report.csv`
7. trained PPO checkpoint + metrics CSV

**Acceptance criteria**
- Real vs simulated distributions are close on core metrics
- PPO baseline beats simple heuristic baseline on at least one held-out split

---

## 1) Recreate and freeze graph-derived priors (from process_graph)
**Why:** Downstream feature engineering needs stable stage/order signals and branch hints.

**Cells to add**
1. Load logs and rebuild minimal shared objects (`logs`, `dfgs`, `act_ranks`)
2. Compute shared backbone transitions (intersection / high-support edges)
3. Compute branch/rare transitions (low-support or municipality-specific)
4. Persist priors to disk (`graph_priors.json`)

**Checks**
- At least one non-empty backbone set
- Rare transitions include meaningful deviations (not only noise)

---

## 2) Build the case-step feature table (one row per event-in-case)
**Why:** This is the RL state table.

**Target schema (minimum)**
- `municipality`, `case_id`, `event_id`, `timestamp`
- `activity`, `prev_activity`, `next_activity`
- `step_index`, `trace_length`, `stage_rank`
- `time_since_case_start_hours`, `time_since_prev_hours`
- `rework_count_activity`, `seen_activity_before`
- branch flags: `is_refusal_path`, `is_suspension_path`, `is_completeness_path`
- outcome helpers: `is_terminal_event`, `case_completed`

**Cells to add**
1. Build `Application -> ordered events` extractor
2. Row-expansion function per case
3. Concatenate all municipalities into one dataframe
4. Type-cleaning + null handling + sort keys
5. Save `case_step_features.parquet`

**Checks**
- No duplicate key on (`municipality`,`case_id`,`event_id`)
- Time deltas are non-negative
- `step_index` spans `0..trace_length-1` per case

---

## 3) Compute duration and branch statistics for simulator calibration
**Why:** Simulator parameters must be learned from data, not guessed.

**Cells to add**
1. Transition probabilities by municipality and global
2. Activity duration distributions (median, IQR, log-mean if skewed)
3. Branch probabilities from designated decision activities
4. Case-arrival proxy from first-event timestamps
5. Persist as `transition_stats.csv`, `duration_stats.csv`, `sim_calibration.json`

**Checks**
- Transition rows sum to ~1 per source activity
- No empty duration model for frequent activities

---

## 4) Define a small, explicit action space
**Why:** PPO requires clear, enforceable decisions.

**Initial actions**
- `continue_normal_path`
- `prioritize_case`
- `request_missing_info`
- `send_to_review`
- `escalate`
- `close_case`

**Cells to add**
1. Action dictionary and integer mapping
2. State-dependent validity rules (action mask function)
3. Export/refresh `valid_action_space.csv` with rule descriptions

**Checks**
- Every state has at least one valid action
- Invalid actions are deterministic from state features

---

## 5) Specify reward function (baseline)
Use a simple linear reward:

$$
r_t = -\alpha\,\text{delay}_t - \beta\,\text{rework}_t - \delta\,\text{invalid}_t + \gamma\,\text{completion}_t
$$

**Cells to add**
1. Parameter block (`alpha, beta, delta, gamma`)
2. Reward component function returning per-step breakdown
3. Sanity plots of reward distributions by municipality

**Checks**
- Completed cases have higher cumulative reward than stalled cases (median)
- Invalid-action penalty dominates tiny delay differences

---

## 6) Build a calibrated SimPy environment
**Why:** PPO needs many trajectories; simulator must mimic real workflow behavior.

**Cells to add**
1. Case generator using arrival-rate estimates
2. Event progression using transition probabilities + duration model
3. Hook for agent action to influence routing/priority
4. Episode termination and logging to dataframe

**Simulator output schema**
- case trace table compatible with `case_step_features` core columns
- episode summary table: duration, loops, terminal status, action counts

**Checks**
- No impossible transitions outside allowed transition table
- Episode always terminates within max step cap

---

## 7) Validate simulator against real logs (must pass before PPO)
**Comparison metrics**
1. Trace length distribution
2. Activity frequency distribution
3. Transition frequency matrix distance
4. Case duration distribution
5. Loop/rework frequency

**Cells to add**
1. Metric computation real vs simulated
2. Side-by-side plots
3. Summary table + pass/fail thresholds
4. Save `sim_validation_report.csv`

**Suggested thresholds (baseline)**
- Relative error < 20% for top activities
- Duration median error < 25%
- Loop frequency error < 20%

---

## 8) Create RL environment wrapper (Gymnasium-style)
**Observation**
- Manual feature vector from case-step row + small history signals

**Action**
- Discrete index over action space with action masking

**Episode end**
- Case completed, dropped, or max-steps reached

**Cells to add**
1. `Env` class (`reset`, `step`, `action_mask`)
2. Vectorized observation builder
3. Random-policy smoke test (10–20 episodes)

**Checks**
- Shapes/dtypes stable
- No NaN in observations/rewards
- Mask blocks invalid actions consistently

---

## 9) Train baseline PPO
**Cells to add**
1. Train/validation split strategy (e.g., municipality hold-out)
2. PPO config (small MLP, conservative learning rate)
3. Training loop + checkpoints
4. Evaluation against heuristic policy

**Minimum experiment matrix**
- Exp A: train on 1–4, test on 5
- Exp B: rotate held-out municipality
- Exp C: synthetic variant stress test

**Outputs**
- `ppo_metrics.csv`
- `ppo_checkpoint/`
- evaluation summary table

---

## 10) Reporting and decision gate for GNN
**Cells to add**
1. Consolidated results table
2. Error analysis: where policy fails (branch type/stage)
3. Decision gate: only proceed to GNN if transfer/generalization plateaus

**Proceed to GNN only if**
- PPO baseline is stable
- Simulator validation is acceptable
- Transfer gap remains significant

---

## 11) Suggested notebook cell order (copy into implementation notebook)
1. Setup/imports/config
2. Load OCEL + reusable helpers
3. Graph priors extraction
4. Case-step feature engineering
5. Stats for calibration
6. Action space + reward
7. SimPy simulator
8. Simulator validation
9. Gym env wrapper
10. PPO training
11. Evaluation + report

---

## Definition of done (for this phase)
- Feature table and calibration artifacts are persisted
- Simulator validated against real data with documented errors
- PPO baseline trained and evaluated on held-out municipality
- Clear go/no-go recommendation for adding GNN embeddings next